# (GR)MHD tutorial

In this notebook we develop the basic concepts and algorithms behind a **conservative finite volume scheme** for magnetohydrodynamics (MHD). We proceed sequentially, building the ideas up one at a time.

We begin with the **1D linear advection problem**, which introduces the spatial discretization and the idea of reconstruction in their simplest setting, free of the complications of a full physical system. 

We then move on to MHD and study a classic 1D test, the **Brio-Wu shock tube**, a Riemann problem whose solution contains fast and slow magnetosonic waves, a contact, and a compound wave, and is a standard benchmark for how faithfully a scheme captures shocks, contact discontinuities, and compound waves. In this context we will understand what are primitive and conserved variables, write down the nonrelativistic ideal MHD equations, and discuss ghost zones for boundary conditions, the Riemann problem at each cell face, approximate Riemann solvers such as the HLL flux (which avoids solving the exact Riemann problem by estimating the flux from the fastest left- and right-going wave speeds), the CFL condition, and time integration. 

Finally, we consider a multidimensional problem, the **Orszag-Tang vortex**, which develops from smooth initial data into a network of interacting shocks and turbulent, small-scale structure, and thus tests a scheme's ability to handle multidimensional MHD dynamics and to maintain a divergence-free magnetic field.

In [66]:
import time
import numpy as np
import matplotlib.pyplot as plt

try:
    import ipywidgets as W
    HAVE_WIDGETS = True
except ImportError:
    HAVE_WIDGETS = False
    print("ipywidgets not found: interactive cells fall back to plain function calls.")

plt.rcParams.update({"figure.dpi": 100, "image.origin": "lower", "font.size": 10})

## 1. Finite volume setup (1D advection)

Consider 1D linear advection with constant speed $a$,
$$q_t + a\,q_x = 0,$$
which is a conservation law $q_t + F(q)_x = 0$ with flux $F(q) = a\,q$.

Divide the domain into cells of width $\Delta x$, with cell $i$ spanning $[x_{i-1/2}, x_{i+1/2}]$. The finite volume unknown is the cell average rather than a point value,
$$\bar q_i = \frac{1}{\Delta x}\int_{x_{i-1/2}}^{x_{i+1/2}} q\,dx.$$
Integrating the equation over the cell yields an exact balance,
$$\frac{d\bar q_i}{dt} + \frac{F_{i+1/2} - F_{i-1/2}}{\Delta x} = 0,$$
in which nothing has been approximated. Only the averages $\{\bar q_i\}$ are stored, whereas the face fluxes $F_{i\pm 1/2}$ require values of $q$ at the faces. Reconstructing a profile $q(x)$ from the stored averages is therefore necessary, and this step is the reconstruction.

### The donor cell reconstruction

The donor cell reconstruction assumes $q$ is constant within each cell, equal to its average,
$$q(x) = \bar q_i, \qquad x\in[x_{i-1/2},x_{i+1/2}].$$

This piecewise constant profile carries no information about the variation of $q$ inside a cell. Because it is discontinuous at every face, each interface has two states, one from each side,
$$q^L_{i+1/2} = \bar q_i, \qquad q^R_{i+1/2} = \bar q_{i+1}.$$

With two competing states at the face, we must decide which one determines the flux through it. The guiding physical idea is that information in an advection problem travels in the direction of the flow: a signal at speed $a$ propagates to the right if $a>0$ and to the left if $a<0$. The value arriving at a face is therefore carried from whichever side lies upstream, and the flux should be built from that side. Taking the state from the upstream cell in this way is called upwinding, and the cell supplying it is the donor cell, since it donates material across the face. For $a>0$ the flow comes from the left, so the upstream cell is cell $i$; for $a<0$ it comes from the right, so the upstream cell is cell $i+1$.

This gives the flux rule, with local propagation speed $a = F'(q)$,
$$F_{i+1/2} = \begin{cases} F(\bar q_i) & a > 0 \quad (\text{donor is cell } i)\\[4pt] F(\bar q_{i+1}) & a < 0 \quad (\text{donor is cell } i+1). \end{cases}$$

For linear advection, $F(q) = a\,q$, this is first order upwinding, which can be written without the case split as
$$F_{i+1/2} = a^{+}\,\bar q_i + a^{-}\,\bar q_{i+1}, \qquad a^{\pm} = \frac{a \pm |a|}{2}.$$

An equivalent rearrangement separates a central average from a correction,
$$F_{i+1/2} = \underbrace{\tfrac{1}{2}\,a\,(\bar q_i + \bar q_{i+1})}_{\text{central average}} \;-\; \underbrace{\tfrac{1}{2}\,|a|\,(\bar q_{i+1} - \bar q_i)}_{\text{added diffusion}},$$
so that first order upwinding is a centered flux plus a term proportional to the face jump. The remainder of this section shows that this correction term is the source of a numerical diffusion.

#### Truncation error analysis

We determine the continuous equation that the donor cell scheme actually approximates by expanding its truncation error. This reveals a spurious diffusion term absent from the original advection equation.

Semi-discrete update. For $a>0$ the donor cell flux is $F_{i+1/2} = a\,\bar q_i$, and the update is
$$\frac{d\bar q_i}{dt} = -\frac{1}{\Delta x}\left(F_{i+1/2}-F_{i-1/2}\right) = -\frac{a}{\Delta x}\left(\bar q_i - \bar q_{i-1}\right).$$

Decomposition of the flux into an average and a difference part.
$$F_{i+1/2} = \underbrace{\tfrac{a}{2}\left(\bar q_i + \bar q_{i+1}\right)}_{\text{central average}} \;-\; \underbrace{\tfrac{|a|}{2}\left(\bar q_{i+1} - \bar q_i\right)}_{\text{correction on the face jump}}.$$
The correction term contains the face jump $\bar q_{i+1}-\bar q_i = q^R_{i+1/2}-q^L_{i+1/2}$, which is nonzero only because the reconstruction is discontinuous at the face.

Flux divergence. The average parts telescope, and the jump corrections combine into a second difference,
$$\frac{d\bar q_i}{dt} = -\frac{a}{2\Delta x}\left(\bar q_{i+1}-\bar q_{i-1}\right) + \frac{|a|}{2\Delta x}\left(\bar q_{i+1}-2\bar q_i+\bar q_{i-1}\right).$$

Identification of the stencils.

$$\frac{\bar q_{i+1}-\bar q_{i-1}}{2\Delta x}\approx q_x, \qquad \frac{\bar q_{i+1}-2\bar q_i+\bar q_{i-1}}{\Delta x^2}\approx q_{xx},$$

so that

$$\frac{d\bar q_i}{dt} \approx -a\,q_x + \frac{|a|\,\Delta x}{2}\,q_{xx}.$$

The scheme is therefore a consistent approximation not of pure advection but of the advection-diffusion equation

$$q_t + a\,q_x = \frac{|a|\,\Delta x}{2}\,q_{xx},$$
with numerical diffusivity $$\nu_{\text{num}} = \frac{|a|\,\Delta x}{2}.$$

This diffusion is not present in the original equation. Since diffusion acts through curvature $q_{xx}$, it smears sharp fronts into gradual ramps and reduces the amplitude of smooth peaks. The error term $\tfrac{|a|\,\Delta x}{2}\,q_{xx}$ carries a **single power of $\Delta x$ and so the method is first order accurate**. Regions where $q_{xx}=0$ are unaffected, so the interior of a plateau is preserved while a Gaussian peak is reduced.

### Piecewise linear reconstruction

The piecewise constant reconstruction limits the scheme to first order because it discards all variation of $q$ within a cell. The natural improvement is to let $q$ vary linearly across each cell, reconstructing a slope in addition to the average, and then to apply the same upwind flux to the reconstructed face values. This is the basis of the MUSCL approach (Monotone Upstream-centered Scheme for Conservation Laws, van Leer 1979). The word "monotone" in the name refers to a property secured by the slope limiters introduced in the next section; here we treat the unlimited version, which is second order but, as we will see, not monotone.

#### The reconstruction

We replace the constant profile by a linear one in each cell,
$$q(x) = \bar q_i + s_i\,(x - x_i), \qquad x\in[x_{i-1/2},x_{i+1/2}],$$
with $x_i$ the cell centre and $s_i$ a slope to be chosen. The cell average is preserved, since the linear term integrates to zero over the cell. Evaluating the profile at an interface $x_{i+1/2}$ gives two states that now carry a slope contribution, the left from cell $i$ and the right from cell $i+1$,
$$q^L_{i+1/2} = \bar q_i + \frac{\Delta x}{2}\,s_i, \qquad q^R_{i+1/2} = \bar q_{i+1} - \frac{\Delta x}{2}\,s_{i+1}.$$
Setting $s_i = 0$ recovers the donor cell states, so the slope is precisely the new ingredient. The upwind flux rule is unchanged; for $a>0$ it selects the left (upstream) state,
$$F_{i+1/2} = a\,q^L_{i+1/2} = a\left(\bar q_i + \frac{\Delta x}{2}\,s_i\right).$$

#### Choice of slope (unlimited)

The simplest second-order choice is the centered slope, which estimates the gradient symmetrically from the two neighbors,
$$s_i = \frac{\bar q_{i+1} - \bar q_{i-1}}{2\Delta x}.$$
The one-sided alternatives, the forward slope $(\bar q_{i+1}-\bar q_i)/\Delta x$ and the backward slope $(\bar q_i - \bar q_{i-1})/\Delta x$, give the closely related Lax-Wendroff and Beam-Warming schemes; all are second order and the conclusions below are unchanged, so we take the centered slope as representative. Because this slope is a fixed linear combination of the data, i.e., the coefficients for the stencil $\bar q_{i+1}, \bar q_{i}, \bar q_{i-1}$ depend on the grid and independent of the solution itself, the scheme is called unlimited.

#### Truncation error analysis for the 'unlimited' linear reconstruction

We repeat the analysis carried out for donor cell. Writing the upwind flux in the same central-plus-dissipation form, now with the reconstructed states,
$$F_{i+1/2} = \underbrace{\tfrac{a}{2}\left(q^L_{i+1/2} + q^R_{i+1/2}\right)}_{\text{central average}} \;-\; \underbrace{\tfrac{|a|}{2}\left(q^R_{i+1/2} - q^L_{i+1/2}\right)}_{\text{dissipation on the face jump}},$$
the dissipation is again proportional to the face jump. Its magnitude is what set the numerical diffusion, so we evaluate it. Substituting the reconstructed states,
$$q^R_{i+1/2} - q^L_{i+1/2} = \left(\bar q_{i+1} - \bar q_i\right) - \frac{\Delta x}{2}\left(s_i + s_{i+1}\right),$$
and inserting the centered slopes, the first difference and the slope term partly cancel, leaving a third difference,
$$q^R_{i+1/2} - q^L_{i+1/2} = -\tfrac14\left(\bar q_{i+2} - 3\bar q_{i+1} + 3\bar q_i - \bar q_{i-1}\right).$$
Identifying the stencil,
$$\frac{\bar q_{i+2} - 3\bar q_{i+1} + 3\bar q_i - \bar q_{i-1}}{\Delta x^3} \approx q_{xxx},$$
so that
$$q^R_{i+1/2} - q^L_{i+1/2} \approx -\frac{\Delta x^3}{4}\,q_{xxx} = O(\Delta x^3).$$

For donor cell the face jump was $O(\Delta x)$ and produced the diffusion term $\tfrac{|a|\Delta x}{2}q_{xx}$. The slope has reduced the jump to $O(\Delta x^3)$: the two reconstructed states now nearly agree at the face, and the even, dissipative $q_{xx}$ term is eliminated. The leading error is no longer diffusive.

Carrying the expansion of the central term to the next order, the reconstruction now reproducing the face value to $O(\Delta x^2)$, the scheme is found to approximate
$$q_t + a\,q_x = \frac{a\,\Delta x^2}{12}\,q_{xxx}.$$
(The precise value of the constant depends on the distinction between the cell average and the point value, and on the slope chosen; the structure, an $O(\Delta x^2)$ third-derivative term, does not.) The error term $\tfrac{a\Delta x^2}{12}q_{xxx}$ carries **two powers of $\Delta x$, so the method is second order accurate**.

#### Godunov's theorem

A scheme whose updated value is a fixed linear combination of the data, $\bar q_i^{\,\text{new}} = \sum_k c_k\,\bar q_{i+k}$ with coefficients $c_k$ independent of the solution, is called a linear scheme; donor cell and the unlimited piecewise linear schemes are both of this type. Godunov's theorem states that **a linear scheme which is monotonicity preserving (does not generate new extrema) can be at most first order accurate**. The two properties we would like, second-order accuracy and the absence of spurious oscillations, are therefore mutually exclusive for any linear scheme.

The reason is intuitive. If the coefficients $c_k$ are all nonnegative and sum to one, the updated value is a weighted average of neighboring values and so cannot exceed their range, which prevents new extrema. However, matching the Taylor series to second order forces at least one coefficient to be negative, and a negative coefficient is precisely what permits an overshoot. In case of donor cell reconstruction, its coefficients are nonnegative, and therefore creates no oscillations, and is first order. The unlimited piecewise linear scheme, however, is second order, and so, by the theorem, it must admit oscillations (see the advected profile for the top hat in the plot below).

The escape is to abandon linearity. A slope limiter makes the reconstructed slope depend on the solution, so that the effective coefficients vary from cell to cell; the scheme is then nonlinear even for the linear advection equation, Godunov's theorem no longer applies, and second-order accuracy in smooth regions can coexist with the suppression of oscillations. This is the subject of the next section.

#### Diffusion versus dispersion

The character of the leading error has changed with its parity. Donor cell's error was an even derivative $q_{xx}$; here it is an odd derivative $q_{xxx}$. Inserting a Fourier mode $q \propto e^{i(kx - \omega t)}$ into each modified equation makes the distinction precise. A diffusion term $\nu\,q_{xx}$ gives the frequency an imaginary part and damps every mode as $e^{-\nu k^2 t}$, reducing amplitude. A term $\mu\,q_{xxx}$ gives only a real correction to the frequency, $\omega = ak + \mu k^3$, so no mode is damped, but the phase speed $\omega/k = a + \mu k^2$ becomes wavelength-dependent. Different wavelengths then travel at slightly different speeds, and a sharp profile, which is a superposition of many modes held in phase, develops spurious oscillations as that phasing drifts apart. This is dispersion rather than diffusion: the unlimited scheme does not smear a discontinuity, but it produces overshoots and undershoots near it, which can drive $q$ outside its physical range. Suppressing these oscillations while retaining second-order accuracy in smooth regions is the purpose of the slope limiters introduced next.

### Slope-limited reconstruction

The two reconstructions so far sit at opposite extremes. Donor cell is free of oscillations but only first order, carrying an $O(\Delta x)$ diffusion that smears every curved feature. The unlimited piecewise linear scheme is second order and non-diffusive in smooth regions, but it produces overshoots and undershoots, driving $q$ outside its physical range near discontinuities. Godunov's theorem showed these outcomes are not accidental: any linear scheme must choose between them. The way past the theorem is to make the reconstruction nonlinear, adapting the slope to the local behaviour of the solution. This is the role of a slope limiter.

#### Total variation

To make precise what we wish to avoid, we measure the total amount of variation in the solution. The total variation of a discrete profile is the sum of its jumps,
$$TV(q) = \sum_i \left|\bar q_{i+1} - \bar q_i\right|.$$

Total variation increases when a new local maximum or minimum appears. The exact solution of a scalar conservation law has nonincreasing total variation in time. A scheme is called total variation diminishing (TVD) if
$$TV(q^{n+1}) \le TV(q^{n})$$
for all data. A TVD scheme cannot manufacture a new oscillation, because a new oscillation would raise the total variation. Donor cell is TVD, which is why it never oscillates; the unlimited scheme is not, which is why it does. The aim of a limiter is to make the second-order piecewise linear scheme TVD.

#### The limited slope

We retain the piecewise linear reconstruction and upwind flux, and modify only the slope. The limiter examines the ratio of the two one-sided differences in each cell,
$$\Delta q^{-}_i = \bar q_i - \bar q_{i-1}, \qquad \Delta q^{+}_i = \bar q_{i+1} - \bar q_i, \qquad r_i = \frac{\Delta q^{-}_i}{\Delta q^{+}_i}.$$
This ratio diagnoses the local behaviour. In a smooth region the two differences are nearly equal and $r_i \approx 1$. At an extremum they have opposite signs and $r_i < 0$, which is the configuration a limiter must catch, since an unlimited slope there would create an overshoot. The limited slope is written as a limiter function $\phi(r)$ times a base slope,
$$s_i = \phi(r_i)\,\frac{\Delta q^{+}_i}{\Delta x}.$$
For the scheme to remain second order in smooth regions the limiter must satisfy $\phi(1) = 1$, so that the limited slope reduces to the centered slope there. For the scheme to be TVD the limiter must vanish at extrema, $\phi(r) = 0$ for $r \le 0$, forcing the slope to zero and the reconstruction locally back to donor cell. A limiter is thus a switch: it keeps the full second-order slope where the solution is smooth and monotone, and reduces the slope toward zero near extrema and discontinuities.

#### Truncation error analysis for the minmod limiter

We repeat the analysis once more. The flux is written in the same central-plus-dissipation form,
$$F_{i+1/2} = \underbrace{\tfrac{a}{2}\left(q^L_{i+1/2} + q^R_{i+1/2}\right)}_{\text{central average}} \;-\; \underbrace{\tfrac{|a|}{2}\left(q^R_{i+1/2} - q^L_{i+1/2}\right)}_{\text{dissipation on the face jump}},$$
so once again the dissipation is proportional to the face jump, and we evaluate it. The reconstructed states are unchanged from the piecewise linear scheme,
$$q^R_{i+1/2} - q^L_{i+1/2} = \left(\bar q_{i+1} - \bar q_i\right) - \frac{\Delta x}{2}\left(s_i + s_{i+1}\right),$$
but the slopes $s_i$ are now the minmod-limited slopes rather than the centered ones. The behaviour depends on which regime the cell is in, so we treat the two cases separately.

Smooth monotone region. Where $q$ varies smoothly and monotonically, the two one-sided differences share a sign, and minmod returns the one of smaller magnitude. Suppose $q_x>0$ and $q_{xx}>0$, so both differences are positive and the backward one is smaller. Then in each cell minmod selects the backward difference,
$$s_i\,\Delta x = \bar q_i - \bar q_{i-1}, \qquad s_{i+1}\,\Delta x = \bar q_{i+1} - \bar q_i.$$
Substituting these into the face jump, the slope term becomes a combination of neighboring differences,
$$q^R_{i+1/2} - q^L_{i+1/2} = \left(\bar q_{i+1} - \bar q_i\right) - \frac{1}{2}\left[\left(\bar q_i - \bar q_{i-1}\right) + \left(\bar q_{i+1} - \bar q_i\right)\right] = \frac{1}{2}\left(\bar q_{i+1} - 2\bar q_i + \bar q_{i-1}\right).$$
Identifying the stencil,
$$\frac{\bar q_{i+1} - 2\bar q_i + \bar q_{i-1}}{\Delta x^2} \approx q_{xx},$$
so that
$$q^R_{i+1/2} - q^L_{i+1/2} \approx \frac{\Delta x^2}{2}\,q_{xx} = O(\Delta x^2).$$

For donor cell the face jump was $O(\Delta x)$; for the unlimited scheme it was $O(\Delta x^3)$; for minmod it is $O(\Delta x^2)$, between the two. Because minmod takes the shallower slope rather than the centered one, it does not close the face jump as completely as the unlimited scheme, so it retains more dissipation. The jump is nonetheless smaller than donor cell's by one power of $\Delta x$, and it is proportional to $q_{xx}$, whose smooth variation contributes a $q_{xxx}$ term to the update rather than a $q_{xx}$ diffusion. The scheme is therefore second order and non-diffusive in this region, though more dissipative than the unlimited scheme.

Extremum. At a smooth maximum $q_x$ passes through zero, so the two one-sided differences $\bar q_i - \bar q_{i-1}$ and $\bar q_{i+1} - \bar q_i$ have opposite signs. Minmod returns zero, the slope vanishes,
$$s_i = 0, \qquad q^R_{i+1/2} - q^L_{i+1/2} = \bar q_{i+1} - \bar q_i = O(\Delta x),$$
and the face jump reverts to the donor cell value. The reconstruction is locally piecewise constant, and the scheme reduces to donor cell with its diffusion $\tfrac{|a|\Delta x}{2}q_{xx}$. The accuracy drops to first order in a neighborhood of every extremum.

The two cases together give both the intended behaviour and its cost. Away from extrema the face jump is $O(\Delta x^2)$ and the scheme is second order, more dissipative than the unlimited scheme but far less than donor cell. At an extremum the slope is clipped to zero, the jump returns to $O(\Delta x)$, and the scheme is locally first order and diffusive. This clipping is forced by the requirement that the limiter vanish for $r \le 0$: a limiter cannot tell a genuine smooth peak from an incipient overshoot from the neighboring gradients alone, so it must clip both. This is the mechanism that reduced the amplitude of the minmod Gaussian in the experiment.

#### Comparison of limiters

The notebook considers three limiters, which differ in how much slope they reconstruct in smooth monotone regions. Minmod always takes the shallower of the two one-sided slopes, the most cautious choice, which makes it the most robust and the most diffusive: it smears discontinuities over the most cells and reduces smooth peaks the fastest. Van Leer and MC take a steeper slope where the data permit, closer to the centered slope, so their face jump and hence their numerical dissipation are smaller in smooth regions. They therefore keep discontinuities sharper and preserve smooth peaks better than minmod, at essentially the same cost. MC in particular uses the centered slope whenever it is admissible, making it the least diffusive of the three and a common default in astrophysical and MHD codes, while van Leer is a smooth function of $r$ and so avoids the abrupt slope changes of the other two. All three, being TVD, still clip smooth extrema to first order and so lose the Gaussian's peak to some degree, differing only in how much.

#### Summary of the various schemes 

Every scheme in this section uses the same reconstruction $s_i = \phi(r_i)\,\Delta q^{+}_i/\Delta x$ and differs only in the limiter $\phi$, with $r_i = \Delta q^{-}_i/\Delta q^{+}_i$. Donor cell ($\phi=0$) and the unlimited scheme ($\phi$ = constant) both give slopes that are fixed linear combinations of the data; the limiters are nonlinear in $r$.

| scheme | $\phi(r)$ | resulting slope $s_i$ | Order (in smooth flows) |
|---|---|---|---|
| donor cell | $0$ | $0$ | first (diffusive) |
| unlimited | $\tfrac{1+r}{2}$ | centered, $\dfrac{\bar q_{i+1}-\bar q_{i-1}}{2\Delta x}$ | second (dispersive) |
| minmod | $\max\!\big(0,\min(1,r)\big)$ | shallower one-sided slope, or $0$ | second, most diffusive |
| van Leer | $\dfrac{r+\lvert r\rvert}{1+\lvert r\rvert}$ | between shallower and centered | second |
| MC | $\max\!\big(0,\min(2r,\tfrac{1+r}{2},2)\big)$ | centered where admissible | second, least diffusive |

At $r=1$, that is in a smooth monotone region, every limiter gives $\phi=1$ and the slope equals the centered slope, so all the second-order schemes agree there. They differ only where $r$ departs from $1$: as $r\to 0$ or $r<0$, near a steep gradient or an extremum. Donor cell takes $\phi=0$ everywhere and the unlimited scheme takes $\phi=\tfrac{1+r}{2}$ everywhere, ignoring the local ratio entirely, which is why the former is always diffusive and the latter always admits oscillations.

In [67]:
def minmod(a, b):
    return 0.5*(np.sign(a) + np.sign(b))*np.minimum(np.abs(a), np.abs(b))

# Every limiter takes the backward and forward differences and returns a slope.
LIMITERS = {
    "donor":     lambda dm, dp: np.zeros_like(dm),
    "unlimited": lambda dm, dp: 0.5*(dm + dp),
    "minmod":    lambda dm, dp: minmod(dm, dp),
    "vanleer":   lambda dm, dp: np.where(dm*dp > 0., 2.*dm*dp/(dm + dp + 1e-300), 0.),
    "mc":        lambda dm, dp: minmod(minmod(2.*dm, 2.*dp), 0.5*(dm + dp)),
}

### Numerical experiment: advection of a top hat and a Gaussian

We now solve the 1D linear advection equation $q_t + a\,q_x = 0$ on a periodic domain $x\in[0,1]$ with unit speed $a=1$, so that after each period the exact solution returns to its initial state. The initial profile combines two features chosen to probe different aspects of a scheme,
$$q_0(x) = \underbrace{\mathbf{1}_{[0.1,\,0.3]}(x)}_{\text{top hat}} \;+\; \underbrace{\exp\!\left[-\left(\frac{x-0.7}{0.06}\right)^2\right]}_{\text{Gaussian}}.$$

The top hat has two discontinuities that test how a scheme handles sharp jumps. The Gaussian is smooth but sharply peaked, and its amplitude tests how faithfully a scheme preserves a resolved smooth extremum. Because the exact solution is simply the initial profile shifted, the black curve in the plots below is both the initial condition and the exact result after any whole number of periods, so any departure of the numerical solution from it is purely numerical error.

The reconstruction is selected with the `limiter` control: `donor` is the piecewise constant reconstruction, `unlimited` is piecewise linear with the centered slope, and the remaining options are slope-limited piecewise linear schemes. The resolution is set by $N$, the number of cells. Comparing the reconstructions on this single profile makes the trade-offs concrete: the diffusion of donor cell, the dispersion of the unlimited scheme, and the compromise reached by the limited schemes.

In [68]:
%matplotlib inline

# --- serif / LaTeX-style text for all plot elements ---
import matplotlib as mpl
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["cmr10", "Computer Modern Roman", "DejaVu Serif"],
    "mathtext.fontset": "cm",     # Computer Modern math
    "axes.formatter.use_mathtext": True,
    "axes.unicode_minus": False,
    "figure.dpi": 200,
    # "text.usetex": True,        # uncomment if you have a system LaTeX install
})

def advect_1d(N=128, cfl=0.3, periods=1, limiter="mc"):
    dx = 1.0/N
    x = (np.arange(N) + 0.5)*dx
    q0 = np.where((x > 0.1) & (x < 0.3), 1.0, 0.0) + np.exp(-((x - 0.7)/0.06)**2)
    nsteps = max(1, int(np.ceil(periods/(cfl*dx))))
    dt = periods/nsteps

    def dqdt(q):
        dm, dp = q - np.roll(q, 1), np.roll(q, -1) - q
        s = LIMITERS[limiter](dm, dp)               # 1. limited slope in every cell
        qL = np.roll(q + 0.5*s, 1)                  # 2. left state at face i-1/2 (from cell i-1)
        F = 1.0*qL                                  # 3. Riemann problem for a>0: take the left state
        return -(np.roll(F, -1) - F)/dx             # 4. flux difference

    q = q0.copy()
    for _ in range(nsteps):
        qhalf = q + 0.5*dt*dqdt(q)                  # predictor: half step
        q = q + dt*dqdt(qhalf)                      # corrector: full step with midpoint fluxes
        if not np.all(np.isfinite(q)) or np.abs(q).max() > 1e3:
            break
    return x, q0, q

def show_advect_1d(N=128, limiter="mc"):
    x, q0, q = advect_1d(N, limiter=limiter)
    fig, ax = plt.subplots(figsize=(9, 3.4))
    ax.plot(x, q0, "k-", lw=1, label=r"$\mathrm{exact}$")
    ax.plot(x, q, "o-", ms=3, lw=1, color="C3", label=rf"$\mathrm{{{limiter}}}$")
    ax.set_ylim(-0.3, 1.5)
    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$q(x)$")
    # ax.set_title(rf"$N={N}\quad L_1\ \mathrm{{error}}={np.mean(np.abs(q - q0)):.3e}\quad$")
    ax.legend(loc="upper right")
    plt.show()
    plt.close(fig)

if HAVE_WIDGETS:
    W.interact(show_advect_1d,
               N=W.SelectionSlider(options=[32, 64, 128, 256, 512], value=128,
                                   description="N", continuous_update=False),
               limiter=W.Dropdown(options=list(LIMITERS), value="donor", description="limiter"));
else:
    show_advect_1d(limiter="donor"); show_advect_1d(limiter="mc")

interactive(children=(SelectionSlider(continuous_update=False, description='N', index=2, options=(32, 64, 128,…

## 2. MHD: primitive and conserved variables, ghost zones, Riemann problem, CFL condition (Brio and Wu shock tube)

We work in Lorentz-Heaviside units, meaning we set $\mu_0 = 1$ (equivalently $\sqrt{4\pi}$ is absorbed into the definition of $\mathbf{B}$ rather than appearing explicitly). In these units the magnetic pressure is $\tfrac{1}{2}B^2$, the magnetic stress is $B_iB_j$, the Alfven speed is simply $v_A = B/\sqrt{\rho}$, and the total pressure is $p_{tot} = p + \tfrac{1}{2}(B^2)$. The conservation equations are:

$$\partial_t \rho + \nabla\!\cdot\!\left(\rho\mathbf{v}\right) = 0$$

$$\partial_t(\rho\mathbf{v}) + \nabla\!\cdot\!\left(\rho\,\mathbf{v}\mathbf{v} - \mathbf{B}\mathbf{B} + p_{tot}\,\mathbf{I}\right) = 0$$

$$\partial_t \mathbf{B} + \nabla\!\cdot\!\left(\mathbf{v}\mathbf{B} - \mathbf{B}\mathbf{v}\right) = 0$$

$$\partial_t E + \nabla\!\cdot\!\left[(E+p_{tot})\mathbf{v} - \mathbf{B}\,(\mathbf{v}\cdot\mathbf{B})\right] = 0$$

with total energy density
$$E = \frac{p}{\gamma-1} + \tfrac{1}{2}\rho v^2 + \tfrac{1}{2} B^2$$

### State and flux vectors

In 2D, the system of conservation laws is

$$\partial_t \mathbf{U} + \partial_x \mathbf{F} + \partial_y \mathbf{G} = 0$$

$$
\mathbf{U} =
\begin{pmatrix}
\rho \\
\rho v_x \\
\rho v_y \\
\rho v_z \\
B_x \\
B_y \\
B_z \\
E
\end{pmatrix},
\qquad
\mathbf{F} =
\begin{pmatrix}
\rho v_x \\
\rho v_x^2 + p_{tot} - B_x^2 \\
\rho v_x v_y - B_x B_y \\
\rho v_x v_z - B_x B_z \\
0 \\
v_x B_y - B_x v_y \\
v_x B_z - B_x v_z \\
(E+p_{tot})v_x - B_x(\mathbf{v}\cdot\mathbf{B})
\end{pmatrix},
\qquad
\mathbf{G} =
\begin{pmatrix}
\rho v_y \\
\rho v_x v_y - B_x B_y \\
\rho v_y^2 + p_{tot} - B_y^2 \\
\rho v_y v_z - B_y B_z \\
v_y B_x - B_y v_x \\
0 \\
v_y B_z - B_y v_z \\
(E+p_{tot})v_y - B_y(\mathbf{v}\cdot\mathbf{B})
\end{pmatrix}
\qquad
\mathbf{P} =
\begin{pmatrix}
\rho \\
v_x \\
v_y \\
v_z \\
B_x \\
B_y \\
B_z \\
p
\end{pmatrix}
$$

### Conserved to primitive ($\mathbf{U} \to \mathbf{P}$)

Unlike relativistic MHD, this inversion is fully analytic (no root-finding). Writing $\mathbf{U} = (U_1,\dots,U_8)$:

$$\rho = U_1, \qquad v_x = \frac{U_2}{U_1}, \qquad v_y = \frac{U_3}{U_1}, \qquad v_z = \frac{U_4}{U_1}$$

$$B_x = U_5, \qquad B_y = U_6, \qquad B_z = U_7$$

$$p = (\gamma-1)\left[\,U_8 - \frac{U_2^2 + U_3^2 + U_4^2}{2U_1} - \tfrac{1}{2}\!\left(U_5^2 + U_6^2 + U_7^2\right)\right]$$

The bracket is just $E$ minus kinetic and magnetic energy, so $p$ follows directly from the energy definition.

In [ ]:
# Variable indices: same names and ordering as in HARM
RHO, UU, U1, U2, U3, B1, B2, B3 = range(8)
NVAR = 8
NG = 2      # ghost zones per side. Linear reconstruction needs 2

# Arrays are P[variable, i, j] with i along x (axis 1) and j along y (axis 2).

def prim_to_cons(P, gam):
    v2 = P[U1]**2 + P[U2]**2 + P[U3]**2
    b2 = P[B1]**2 + P[B2]**2 + P[B3]**2
    U = np.empty_like(P)
    U[RHO] = P[RHO]
    U[UU] = P[UU] + 0.5*P[RHO]*v2 + 0.5*b2          # total energy
    U[U1:U3+1] = P[RHO]*P[U1:U3+1]                  # momentum
    U[B1:] = P[B1:]
    return U

def cons_to_prim(U, gam, rho_floor=1e-20, u_floor=1e-20):
    P = np.empty_like(U)
    P[RHO] = np.maximum(U[RHO], rho_floor)          # floors: the code must survive a bad zone
    P[U1:U3+1] = U[U1:U3+1]/P[RHO]
    P[B1:] = U[B1:]
    v2 = P[U1]**2 + P[U2]**2 + P[U3]**2
    b2 = P[B1]**2 + P[B2]**2 + P[B3]**2
    P[UU] = np.maximum(U[UU] - 0.5*P[RHO]*v2 - 0.5*b2, u_floor)
    return P

def prim_to_flux(P, d, gam):
    # Flux of every conserved variable in direction d (1 = x, 2 = y)
    vn, Bn = P[U1+d-1], P[B1+d-1]
    v2 = P[U1]**2 + P[U2]**2 + P[U3]**2
    b2 = P[B1]**2 + P[B2]**2 + P[B3]**2
    vB = P[U1]*P[B1] + P[U2]*P[B2] + P[U3]*P[B3]
    ptot = (gam - 1.)*P[UU] + 0.5*b2
    E = P[UU] + 0.5*P[RHO]*v2 + 0.5*b2
    F = np.empty_like(P)
    F[RHO] = P[RHO]*vn
    F[UU] = (E + ptot)*vn - Bn*vB
    for k in range(3):
        F[U1+k] = P[RHO]*vn*P[U1+k] - Bn*P[B1+k]
        F[B1+k] = vn*P[B1+k] - Bn*P[U1+k]           # note: this is an electric field (Part 4)
    F[U1+d-1] += ptot
    return F

### 2.3 Reconstruction and ghost zones

The reconstruction is exactly the 1D procedure developed in Part 1. In each cell we build a profile from the stored averages, evaluate it on the two faces to obtain the left and right states, and use that to compute the flux. The only change in the MHD setting is that the recipe is applied to each primitive variable separately, and along each spatial direction in turn. **Face $i$ is the interface at the left edge of cell $i$**, i.e. the interface between cell $i-1$ and cell $i$ (what Part 1 wrote as face $i-1/2$). Its left state is reconstructed from cell $i-1$ and its right state from cell $i$.

#### Why linear reconstruction needs two neighbors

Recall from Part 1 that the piecewise linear reconstruction assigns each cell a slope estimated from its neighbors. With the centered slope,
$$s_i = \frac{\bar q_{i+1} - \bar q_{i-1}}{2\Delta x},$$
the reconstruction in cell $i$ reads both $\bar q_{i-1}$ and $\bar q_{i+1}$, and the limited slopes use the same one-sided differences $\bar q_i - \bar q_{i-1}$ and $\bar q_{i+1} - \bar q_i$. Constructing the profile in cell $i$ therefore requires its two immediate neighbors. This is the price of going beyond the piecewise constant donor cell reconstruction, which needs only the cell itself.

Now consider the outermost physical cell of the grid. Updating it requires the flux on its outer face, and that flux takes its upwind state from a reconstruction in the first cell outside the physical domain. A linear reconstruction in that outside cell needs its own neighbor to form a slope, which lies a second cell beyond the domain. The stencil to compute the slope, thus demands valid data two cells beyond the edge: one to supply the reconstruction on the outermost face, and a second to build that reconstruction's slope. A first-order (donor cell) scheme reconstructs without a slope and so needs a single layer, while a linear scheme needs two layers on each side. These extra cells are the **ghost zones**.

#### Ghost zones and boundary conditions

Ghost zones are additional cells appended outside the physical domain whose sole purpose is to supply the reconstruction stencil with data at the edges, so that the same interior update can be applied uniformly to every physical cell without special-casing the boundaries. They carry no independent physics; their values are set entirely by the boundary condition, which encodes how the solution is meant to behave at the edge of the domain.

The routine `set_bounds` fills the ghost zones from the chosen boundary condition. Two common cases appear in this notebook:

- Periodic: the domain wraps, so a ghost zone on one side is copied from the physical cells on the far side. This is the condition used in the advection tests, where the profile leaves one edge and re-enters the other.
- Outflow (zero-gradient): the ghost zones copy the value of the nearest physical cell, so that gradients vanish at the boundary and waves leave the domain without reflecting back in. This is the natural choice for the Brio-Wu shock tube, whose left and right states extend to infinity in the idealised problem.

Because the ghost zones feed the reconstruction, they must hold valid data every time a reconstruction is performed. `set_bounds` is therefore called after every substep of the time integration, not merely once per step: each substep updates the physical cells and leaves the ghost zones stale, and the next reconstruction would read those stale values unless they are refreshed first. In a parallel code these same ghost zones are the data exchanged between neighboring MPI ranks, each rank treating its neighbor's edge cells as the boundary condition for its own subdomain, which is why a clean ghost zone abstraction is what makes the serial and parallel updates identical.

In [5]:
def reconstruct(P, d, limiter):
    dm = P - np.roll(P, 1, axis=d)
    dp = np.roll(P, -1, axis=d) - P
    s = LIMITERS[limiter](dm, dp)
    Pl = np.roll(P + 0.5*s, 1, axis=d)     # left state at face i  = right edge of cell i-1
    Pr = P - 0.5*s                         # right state at face i = left edge of cell i
    return Pl, Pr
    # (np.roll wraps around at the array edge, but that only pollutes the outermost
    #  ghost zone, whose value is never used.)

def set_bounds(P, bc="periodic"):
    n1, n2 = P.shape[1] - 2*NG, P.shape[2] - 2*NG
    for g in range(NG):                    # x direction (full rows, so corners get filled too)
        if bc == "periodic":
            P[:, g, :], P[:, NG+n1+g, :] = P[:, n1+g, :], P[:, NG+g, :]
        else:                              # outflow: copy last physical zone
            P[:, g, :], P[:, NG+n1+g, :] = P[:, NG, :], P[:, NG+n1-1, :]
    for g in range(NG):                    # y direction
        if bc == "periodic":
            P[:, :, g], P[:, :, NG+n2+g] = P[:, :, n2+g], P[:, :, NG+g]
        else:
            P[:, :, g], P[:, :, NG+n2+g] = P[:, :, NG], P[:, :, NG+n2-1]

### The Riemann problem

The reconstruction leaves two states at every face, $q^L$ from the cell on the left and $q^R$ from the cell on the right, while the update requires a single flux through that face. A rule producing one flux from the two states is called a Riemann solver.

In the advection problem this was trivial: a single wave carries the profile at speed $a$, so information arrives at a face from one side only, and the flux is built from $q^L$ for $a>0$ and from $q^R$ for $a<0$. A gas is different because it supports several disturbances at once. With $\rho$ the density, $p$ the pressure, $v$ the fluid velocity and $c_s = \sqrt{\gamma p/\rho}$ the sound speed, information travels at the three characteristic speeds $v-c_s$, $v$, and $v+c_s$. In subsonic flow $v-c_s<0<v+c_s$, so information reaches the face from both sides and there is no single upwind direction.

Two uniform states meeting at a discontinuity is a classic shock tube problem in gas dynamics, known as the Riemann problem, and the reconstruction creates such a problem at the face between every pair of grid zones. Its solution is a set of waves fanning out from the interface, one per characteristic speed: a sound wave in each direction, which steepens into a shock or spreads into a rarefaction and changes pressure, density and velocity together, and between them a contact discontinuity, across which pressure and velocity match while density jumps. If all the waves move right the face sees $q^L$ and upwinding is recovered, if all move left it sees $q^R$, and if they straddle the face it sees an intermediate state created by the interaction.

MHD has the same structure with more waves, since the magnetic field $\mathbf{B}$ adds restoring forces. Taking $x$ as the direction of propagation, there are seven characteristic speeds:

- the entropy wave at $v_x$, across which density jumps but pressure and velocity do not;
- the Alfvén waves at $v_x \pm c_a$, with $c_a = |B_x|/\sqrt{\rho}$, which consist of transverse perturbations of the magnetic field and are incompressible;
- the fast magnetosonic waves at $v_x \pm c_f$, compressive waves where the gas and the magnetic field are compressed in phase; this is the fastest signal in the system;
- the slow magnetosonic waves at $v_x \pm c_{\text{slow}}$, compressive waves where the gas is compressed as the magnetic field is rarefied, resulting a weaker restoring force.

Solving the MHD Riemann problem exactly is expensive, so approximate solvers are used.

The HLL solver keeps only the fastest left- and right-going waves, with speeds $S_L$ and $S_R$ estimated from the reconstructed states, and replaces everything between them with a single uniform state fixed by conservation. The flux is
$$F^{\text{HLL}} = \begin{cases} F^L & S_L \ge 0\\[6pt] \dfrac{S_R\,F^L - S_L\,F^R + S_L S_R\left(q^R - q^L\right)}{S_R - S_L} & S_L < 0 < S_R\\[10pt] F^R & S_R \le 0.\end{cases}$$
The outer cases are supersonic flow, where the fan moves one way and upwinding applies. The middle case is an average of the two fluxes minus a term proportional to the face jump $q^R-q^L$.

The local Lax-Friedrichs (LLF) flux sets $S_R = -S_L = S$, with $S$ the largest characteristic speed at the face, reducing HLL to
$$F^{\text{LLF}} = \tfrac{1}{2}\left(F^L + F^R\right) - \tfrac{1}{2}\,S\left(q^R - q^L\right),$$
a centered flux with dissipation set by the fastest signal. It needs no case distinction and only one speed, at the cost of applying the fastest wave's dissipation to every wave.

In [ ]:
def wave_speeds(P, d, gam):
    # Fastest right- and left-going signal speeds: v_n +/- c_fast
    a2 = gam*(gam - 1.)*P[UU]/P[RHO]                 # sound speed^2
    b2 = (P[B1]**2 + P[B2]**2 + P[B3]**2)/P[RHO]     # Alfven speed^2
    bn2 = P[B1+d-1]**2/P[RHO]
    cf = np.sqrt(0.5*(a2 + b2 + np.sqrt(np.maximum((a2 + b2)**2 - 4.*a2*bn2, 0.))))
    return P[U1+d-1] + cf, P[U1+d-1] - cf

def riemann(Pl, Pr, d, gam, solver="llf"):
    Fl, Fr = prim_to_flux(Pl, d, gam), prim_to_flux(Pr, d, gam)
    Ul, Ur = prim_to_cons(Pl, gam), prim_to_cons(Pr, gam)
    cpl, cml = wave_speeds(Pl, d, gam)
    cpr, cmr = wave_speeds(Pr, d, gam)
    cmax = np.maximum(0., np.maximum(cpl, cpr))      # lambda^+        (>= 0)
    cmin = np.maximum(0., np.maximum(-cml, -cmr))    # -lambda^-       (>= 0)
    if solver == "llf":
        ctop = np.maximum(cmax, cmin)
        return 0.5*(Fl + Fr - ctop*(Ur - Ul))
    if solver == "hll":
        return (cmax*Fl + cmin*Fr - cmax*cmin*(Ur - Ul))/(cmax + cmin + 1e-300)
    raise ValueError(solver)

## 2.5 Update, time step, and the CFL condition

Once the Riemann solver has supplied a flux on every face, the conserved variables are advanced by differencing those fluxes. In two dimensions the update of cell $(i,j)$ is the finite volume balance from Part 1 with one term per direction,

$$\mathbf{U}_{i,j}^{\,\text{new}} = \mathbf{U}_{i,j} - \frac{\Delta t}{\Delta x}\left(\mathbf{F}_{i+1/2,j} - \mathbf{F}_{i-1/2,j}\right) - \frac{\Delta t}{\Delta y}\left(\mathbf{G}_{i,j+1/2} - \mathbf{G}_{i,j-1/2}\right),$$

which states that the conserved content of a cell changes only through the fluxes crossing its four faces.

### The CFL condition

The time step cannot be chosen freely. The exact solution carries information at the characteristic speeds, and a scheme can only converge if its stencil covers the region the true solution depends on (the Courant-Friedrichs-Lewy, or CFL, condition). For a Godunov-type scheme the practical limit is tighter: each face flux comes from an isolated Riemann problem, which stays valid only if its waves do not reach the neighboring face within one step. The fastest signal must therefore travel less than one cell per step. This is necessary for stability but not sufficient, so in practice the Courant number is kept somewhat below one.

Writing $c_{\text{top}}$ for the fastest signal speed at a face, which in MHD is the fast magnetosonic speed $|v|+c_f$, the condition in two dimensions is

$$\Delta t = C \,\Big/ \max_{\text{zones}}\left(\frac{c_{\text{top},x}}{\Delta x} + \frac{c_{\text{top},y}}{\Delta y}\right), \qquad C < 1.$$

The maximum runs over the whole grid, so the step is set by the single most restrictive cell, typically the hottest or most strongly magnetised one, where $c_f$ is largest. The dimensionless number $C$ is the Courant number; $C<1$ ensures that the fastest wave moves less than one cell per step. The same fast speed computed here for the time step is the one already used to bound the Riemann fan, so the two uses of the wave speeds are consistent: one asks how far information moves globally to set $\Delta t$, the other asks how far it moves locally at a face to set the flux.

### Time integration: from forward Euler to predictor-corrector

The update above tells us the rate of change of the conserved variables, but not yet how to advance them in time. Write the spatially discretised system compactly as

$$\frac{d\mathbf{U}}{dt} = \mathcal{L}(\mathbf{U}),$$

where $\mathcal{L}$ denotes the full right-hand side: reconstruct, solve the Riemann problems, and difference the fluxes. Everything to the left of this point in the notebook has been about building $\mathcal{L}$. What remains is to integrate it in time.

The simplest choice is forward Euler, which advances the solution using the rate of change evaluated at the start of the step,

$$\mathbf{U}^{n+1} = \mathbf{U}^{n} + \Delta t\,\mathcal{L}(\mathbf{U}^{n}).$$

This is first order accurate in time. The reason is that it approximates the solution by the first two terms of its Taylor expansion, $\mathbf{U}(t+\Delta t) = \mathbf{U} + \Delta t\,\mathbf{U}_t + \tfrac{1}{2}\Delta t^2 \mathbf{U}_{tt} + \dots$, and drops everything from the $\Delta t^2$ term onward. The neglected terms are $O(\Delta t^2)$ per step, and accumulated over the $O(1/\Delta t)$ steps needed to reach a fixed time they leave an error of $O(\Delta t)$. Pairing a first-order time integrator with the second-order spatial reconstruction built above would waste that spatial accuracy, since the overall error would be limited by the weaker of the two.

To match the second-order reconstruction we use a two-stage predictor-corrector scheme. It advances in two substeps:

$$\mathbf{U}^{*} = \mathbf{U}^{n} + \tfrac{1}{2}\Delta t\,\mathcal{L}(\mathbf{U}^{n}), \qquad \mathbf{U}^{n+1} = \mathbf{U}^{n} + \Delta t\,\mathcal{L}(\mathbf{U}^{*}).$$

The first substep is a forward Euler half-step used only to produce a provisional state $\mathbf{U}^*$ at the midpoint of the interval, the predictor. The second substep discards the provisional state's role as an answer and keeps only the rate of change evaluated there, $\mathcal{L}(\mathbf{U}^*)$, using that midpoint rate to take the full step from $\mathbf{U}^n$, the corrector. Evaluating the rate at the midpoint rather than at the start cancels the leading time-truncation error, so the scheme is second order accurate in time, matching the spatial order. This is the explicit midpoint method, and it is exactly the two-line `step` routine in the code below: `advance` applied once over half a step to build the predictor, then again over the full step using fluxes computed from the predicted state.

Note that both substeps compute their fluxes from the same predicted or initial state and apply them to the original `U`, so each substep is a complete pass through reconstruction, the Riemann solver, and the flux difference. The ghost zones are refilled at the end of each `advance`, since the next reconstruction will read them.


In [ ]:
def flux_ct(F1, F2):
    # Flux-CT (Toth 2000)
    emf = 0.25*(F1[B2] + np.roll(F1[B2], 1, axis=1)
                - F2[B1] - np.roll(F2[B1], 1, axis=0))       # lives at the cell corner (i-1/2, j-1/2)
    F1[B1] = 0.
    F1[B2] = 0.5*(emf + np.roll(emf, -1, axis=1))
    F2[B1] = -0.5*(emf + np.roll(emf, -1, axis=0))
    F2[B2] = 0.

def divB(P, dx, dy):
    # Corner-centered divergence: the particular stencil that Flux-CT preserves
    bx, by = P[B1], P[B2]
    d = (0.5*(bx + np.roll(bx, 1, 1) - np.roll(bx, 1, 0) - np.roll(np.roll(bx, 1, 0), 1, 1))/dx
       + 0.5*(by + np.roll(by, 1, 0) - np.roll(by, 1, 1) - np.roll(np.roll(by, 1, 0), 1, 1))/dy)
    return d[NG:-NG, NG:-NG]

def advance(Pi, Ps, dt, dx, dy, gam, limiter, solver, ct, bc):
    # Advance state Pi by dt using fluxes computed from state Ps
    Pl, Pr = reconstruct(Ps, 1, limiter)
    F1 = riemann(Pl, Pr, 1, gam, solver)
    Pl, Pr = reconstruct(Ps, 2, limiter)
    F2 = riemann(Pl, Pr, 2, gam, solver)
    if ct:
        flux_ct(F1, F2)
    U = prim_to_cons(Pi, gam)
    U -= dt/dx*(np.roll(F1, -1, axis=1) - F1) + dt/dy*(np.roll(F2, -1, axis=2) - F2)
    Pf = cons_to_prim(U, gam)
    set_bounds(Pf, bc)
    return Pf

def step(P, dt, dx, dy, gam, limiter, solver, ct, bc):
    Phalf = advance(P, P, 0.5*dt, dx, dy, gam, limiter, solver, ct, bc)     # predictor
    return advance(P, Phalf, dt, dx, dy, gam, limiter, solver, ct, bc)      # corrector

def new_dt(P, dx, dy, gam, cour):
    cp1, cm1 = wave_speeds(P, 1, gam)
    cp2, cm2 = wave_speeds(P, 2, gam)
    c1 = np.maximum(np.abs(cp1), np.abs(cm1))
    c2 = np.maximum(np.abs(cp2), np.abs(cm2))
    return cour/np.max(c1/dx + c2/dy)

### The Brio-Wu shock tube

The Brio-Wu problem is the standard one-dimensional test for an MHD scheme, playing a similar role as the Sod problem for hydrodynamics. Two uniform states are placed in contact at the centre of the domain: a dense, high-pressure state on the left and a thin, low-pressure state on the right, with a magnetic field whose transverse component is reversed between the two sides while its normal component is held constant. At $t=0$ the interface is released.

Because the two states are out of balance in both gas pressure and magnetic field, the discontinuity cannot persist and resolves into a fan of waves spreading outward, exactly the Riemann problem of the previous section with concrete initial data. Where the Sod problem produces three features (a rarefaction, a contact, and a shock), the reversed field here excites the magnetic wave families as well, so the solution contains the fast and slow magnetosonic waves, the contact, and a compound wave that has no hydrodynamic counterpart.

The panels below show the primitive variables across the domain at a chosen time. Reading the profile from left to right recovers the individual waves of the fan, and comparing reconstructions and Riemann solvers shows how each resolves the discontinuities and the compound wave.

In [ ]:
# Brio-Wu shock tube: 1D MHD Riemann problem with a jump at x=0.5, solved in a 2D grid with periodic y.

def init_briowu(N, gam=2.0):
    """Brio-Wu MHD shock tube on x in [0,1] with a jump at x=0.5.
       We embed the 1D problem as an N x 4 grid, y direction unused."""
    Ny = 4
    dx = 1.0/N
    xc = (np.arange(-NG, N + NG) + 0.5)*dx
    P = np.zeros((NVAR, N + 2*NG, Ny + 2*NG))
    left = xc[:, None] < 0.5
    P[RHO] = np.where(left, 1.0,   0.125)
    P[UU]  = np.where(left, 1.0,   0.1) / (gam - 1.0)
    P[B1]  = 0.75
    P[B2]  = np.where(left, 1.0,  -1.0)
    set_bounds(P, bc="outflow")            # outflow in both directions is fine: all rows are identical
    return P, dx

BW_RESULTS = {}

def run_briowu(N=400, tf=0.1, limiter="mc", solver="hll", cour=0.4, gam=2.0, nsnap=21):
    key = (N, tf, limiter, solver, cour, gam, nsnap)
    if key in BW_RESULTS:
        return BW_RESULTS[key]
    P, dx = init_briowu(N, gam)
    dy = dx
    t = 0.0; nstep = 0
    tsnap = np.linspace(0.0, tf, nsnap)
    snaps = [P[:, NG:-NG, NG].copy()]
    tsaved = [0.0]
    isnap = 1
    while t < tf - 1e-12 and isnap < nsnap:
        dt = min(new_dt(P, dx, dy, gam, cour), tsnap[isnap] - t, tf - t)
        P = step(P, dt, dx, dy, gam, limiter, solver, ct=False, bc="outflow")
        t += dt; nstep += 1
        if not np.all(np.isfinite(P)):
            print(f"NaN at t={t:.3f}, stopping"); break
        if t >= tsnap[isnap] - 1e-12:
            snaps.append(P[:, NG:-NG, NG].copy())
            tsaved.append(t)
            isnap += 1
    xc = (np.arange(N) + 0.5)*dx
    BW_RESULTS[key] = dict(x=xc, snaps=snaps, t=np.array(tsaved), gam=gam,
                           label=f"{limiter}, {solver}, N={N}, cour={cour}", nstep=nstep)
    return BW_RESULTS[key]

def show_briowu(N=400, tf=0.1, limiter="mc", solver="hll", cour=0.4, nsnap=21):
    r = run_briowu(N=N, tf=tf, limiter=limiter, solver=solver, cour=cour, nsnap=nsnap)

    def frame(k=len(r["snaps"]) - 1):
        P = r["snaps"][k]; gam = r["gam"]
        rho, vx, vy, By = P[RHO], P[U1], P[U2], P[B2]
        p = (gam - 1.0)*P[UU]
        fig, axes = plt.subplots(5, 1, figsize=(9, 11), sharex=True)
        for ax, y, name in zip(axes, [rho, vx, vy, By, p],
                                [r"$\rho$", r"$v_x$", r"$v_y$", r"$B_y$", r"$p$"]):
            ax.plot(r["x"], y, "-", lw=1, color="C3")
            ax.set_ylabel(name); ax.grid(alpha=0.3)
        # axes[0].set_title(f"Brio-Wu at t = {r['t'][k]:.3f}   |   {r['label']}   |   {r['nstep']} steps total")
        axes[-1].set_xlabel("x")
        fig.tight_layout(); plt.show(); plt.close(fig)

    if HAVE_WIDGETS:
        W.interact(frame,
                   k=W.IntSlider(min=0, max=len(r["snaps"]) - 1,
                                 value=len(r["snaps"]) - 1,
                                 description="snapshot",
                                 continuous_update=False))
    else:
        frame()

if HAVE_WIDGETS:
    W.interact_manual(show_briowu,
                      N=W.SelectionSlider(options=[100, 200, 400, 800, 1600], value=800,
                                          description="N"),
                      tf=W.FloatSlider(min=0.02, max=0.2, step=0.01, value=0.2,
                                       description="t_final", continuous_update=False),
                      limiter=W.Dropdown(options=list(LIMITERS), value="mc", description="limiter"),
                      solver=W.Dropdown(options=["llf", "hll"], value="llf", description="Riemann"),
                      cour=W.FloatSlider(min=0.1, max=1., step=0.05, value=0.5,
                                         description="CFL", continuous_update=False),
                      nsnap=W.IntSlider(min=6, max=51, step=5, value=21,
                                        description="snapshots", continuous_update=False));
else:
    show_briowu(N=800, limiter="mc", solver="llf")

interactive(children=(SelectionSlider(description='N', index=3, options=(100, 200, 400, 800, 1600), value=800)…

## 3. Keeping the magnetic field divergence-free (Orszag-Tang vortex to come)

Everything so far treated the magnetic field like any other conserved variable: reconstruct it, solve a Riemann problem, difference the fluxes. In one dimension that is enough. In two or more dimensions it is not, because the magnetic field carries an extra constraint that the ordinary finite volume update does not respect.

### The constraint

Maxwell's equations require the magnetic field to be divergence-free at all times,

$$\nabla\cdot\mathbf{B} = \frac{\partial B_x}{\partial x} + \frac{\partial B_y}{\partial y} = 0,$$

which physically states that there are no magnetic monopoles: field lines have no sources or sinks. The induction equation preserves this exactly in the continuum, so if the field starts divergence-free it stays divergence-free forever.

The discrete scheme does not inherit this automatically. When each field component is updated by an independent flux difference, like the density or momentum, the particular combination of derivatives that makes up $\nabla\cdot\mathbf{B}$ is not conserved by the update, and a small nonzero divergence is generated every step. A nonzero $\nabla\cdot\mathbf{B}$ corresponds to unphysical magnetic monopoles on the grid, which exert spurious forces along the field, accelerate the plasma in the wrong direction, and can grow until the solution is corrupted or the run fails. Controlling $\nabla\cdot\mathbf{B}$ is therefore a genuine requirement of a multidimensional MHD code.

Several strategies exist. The approach used here, and in HARM, is to construct the update so that a particular discrete measure of $\nabla\cdot\mathbf{B}$ is preserved to round-off from the start, and never allowed to grow. This is called constrained transport.

### Flux-interpolated constrained transport

Constrained transport enforces the constraint by rewriting the induction equation as a curl and building the update so that its discrete divergence vanishes identically. Faraday's law is

$$\frac{\partial \mathbf{B}}{\partial t} = -\,\nabla\times\mathbf{E} = -\,\nabla\times\left(-\,\mathbf{v}\times\mathbf{B}\right) = \nabla\times\left(\mathbf{v}\times\mathbf{B}\right),$$

where the last equality uses the ideal-MHD electric field $\mathbf{E} = -\,\mathbf{v}\times\mathbf{B}$. Taking the divergence,

$$\frac{\partial (\nabla\cdot\mathbf{B})}{\partial t} = \nabla\cdot\left[\nabla\times\left(\mathbf{v}\times\mathbf{B}\right)\right] = 0,$$

because the divergence of a curl is identically zero. So a field that starts divergence-free stays divergence-free, provided the field is advanced by a genuine curl. The ordinary flux-difference update does not have this property: it advances $B_x$ and $B_y$ by two independent flux differences that are not tied together as a curl, and the diffusive Riemann fluxes then generate a nonzero $\nabla\cdot\mathbf{B}$.

Flux-CT (Toth 2000), used in HARM, repairs this without changing the update itself. It replaces the magnetic fluxes returned by the Riemann solver with fluxes that difference to the discrete curl of $\mathbf{v}\times\mathbf{B}$, so that when the generic update runs, the field is advanced by a curl and the discrete divergence is preserved to machine precision. The replacement is done in two steps.

First, the EMF is assembled at the cell corner $(i-\tfrac12, j-\tfrac12)$ by averaging the transverse-field fluxes from the four faces meeting there,

$$\Omega_{i-\frac12,\,j-\frac12} = \tfrac14\left[\,F^{x}_{i,j}(B_y) + F^{x}_{i,j-1}(B_y) - F^{y}_{i,j}(B_x) - F^{y}_{i-1,j}(B_x)\,\right],$$

where $F^{x}_{i,j}(B_y)$ is the flux of $B_y$ in the $x$-direction at zone $(i,j)$, and $F^{y}_{i,j}(B_x)$ the flux of $B_x$ in the $y$-direction. These are exactly the induction fluxes already computed by the Riemann solver, since $F^{x}(B_y) = v_x B_y - v_y B_x$ and $F^{y}(B_x) = v_y B_x - v_x B_y$ are the two representations of the same electric field, up to sign. The corner EMF $\Omega$ is their four-face average.

Second, the four magnetic flux entries are reset from this corner EMF. The normal-field fluxes are zeroed, and the transverse-field fluxes are set to the average of the two corner EMFs bounding each face,

$$F^{x}_{i,j}(B_x) = 0, \qquad F^{x}_{i,j}(B_y) = \tfrac12\left[\,\Omega_{i-\frac12,\,j-\frac12} + \Omega_{i-\frac12,\,j+\frac12}\,\right],$$

$$F^{y}_{i,j}(B_y) = 0, \qquad F^{y}_{i,j}(B_x) = -\tfrac12\left[\,\Omega_{i-\frac12,\,j-\frac12} + \Omega_{i+\frac12,\,j-\frac12}\,\right].$$

The normal fluxes vanish because a field component is not transported in its own direction. The transverse fluxes carry the corner EMF, interpolated from the two corners on each face; the opposite signs on the $x$- and $y$-direction fluxes are the two components of the curl, and they are what make the mixed differences cancel in the divergence.